In [ ]:
import pandas as pd

# -----------------------------
# 1. 데이터 로드
# -----------------------------
orders = pd.read_csv('../data/raw/orders.csv')
prior = pd.read_csv('../data/raw/order_products__prior.csv')
train = pd.read_csv('../data/raw/order_products__train.csv')
products = pd.read_csv('../data/raw/products.csv')
aisles = pd.read_csv('../data/raw/aisles.csv')
departments = pd.read_csv('../data/raw/departments.csv')

# -----------------------------
# 2. 상품 메타데이터 병합
# -----------------------------
product_info = products.merge(aisles, on='aisle_id', how='left')
product_info = product_info.merge(departments, on='department_id', how='left')

# -----------------------------
# 3. orders에서 prior / train 주문만 분리
# -----------------------------
prior_orders = orders[orders['eval_set'] == 'prior'][[
    'order_id', 'user_id', 'order_number',
    'order_dow', 'order_hour_of_day', 'days_since_prior_order'
]]
train_orders = orders[orders['eval_set'] == 'train'][[
    'order_id', 'user_id', 'order_number',
    'order_dow', 'order_hour_of_day', 'days_since_prior_order'
]]

# -----------------------------
# 4. prior에 user 정보 붙이기
# -----------------------------
prior = prior.merge(prior_orders, on='order_id', how='left')

# -----------------------------
# 5. train에 user 정보 붙이기
# -----------------------------
train = train.merge(train_orders[['order_id', 'user_id']], on='order_id', how='left')
train['label'] = 1

# -----------------------------
# 6. 유저-상품별 과거 구매 이력 피처 생성
# -----------------------------
user_product_features = prior.groupby(['user_id', 'product_id']).agg(
    up_order_count=('order_id', 'count'),
    up_first_order_num=('order_number', 'min'),
    up_last_order_num=('order_number', 'max'),
    up_avg_add_to_cart=('add_to_cart_order', 'mean'),
    up_reordered_sum=('reordered', 'sum')
).reset_index()

# -----------------------------
# 7. 유저별 전체 주문 수 피처
# -----------------------------
user_features = prior.groupby('user_id').agg(
    user_total_orders=('order_number', 'max'),
    user_total_items=('product_id', 'count'),
    user_total_distinct_items=('product_id', 'nunique'),
    user_avg_days_since_prior=('days_since_prior_order', 'mean')
).reset_index()

# -----------------------------
# 8. 상품별 피처
# -----------------------------
product_features = prior.groupby('product_id').agg(
    product_total_purchases=('order_id', 'count'),
    product_total_reorders=('reordered', 'sum'),
    product_unique_users=('user_id', 'nunique')
).reset_index()

product_features['product_reorder_rate'] = (
    product_features['product_total_reorders'] / product_features['product_total_purchases']
)

# -----------------------------
# 9. 학습용 base 생성
# -----------------------------
base = prior[['user_id', 'product_id']].drop_duplicates()

# -----------------------------
# 10. label 붙이기
# -----------------------------
labels = train[['user_id', 'product_id', 'label']].drop_duplicates()

df = base.merge(labels, on=['user_id', 'product_id'], how='left')
df['label'] = df['label'].fillna(0).astype(int)

# -----------------------------
# 11. 피처들 병합
# -----------------------------
df = df.merge(user_product_features, on=['user_id', 'product_id'], how='left')
df = df.merge(user_features, on='user_id', how='left')
df = df.merge(product_features, on='product_id', how='left')
df = df.merge(product_info, on='product_id', how='left')

# -----------------------------
# 12. 추가 파생변수
# -----------------------------
df['up_order_rate'] = df['up_order_count'] / df['user_total_orders']
df['up_recency'] = df['user_total_orders'] - df['up_last_order_num']
df['up_first_last_gap'] = df['up_last_order_num'] - df['up_first_order_num']

# -----------------------------
# 13. 결측치 처리
# -----------------------------
df['user_avg_days_since_prior'] = df['user_avg_days_since_prior'].fillna(-1)
df['product_reorder_rate'] = df['product_reorder_rate'].fillna(-1)
df['up_avg_add_to_cart'] = df['up_avg_add_to_cart'].fillna(-1)

# -----------------------------
# 14. 용량 최적화
# -----------------------------
# 모델 학습에 꼭 필요하지 않은 문자열 컬럼 제거
drop_cols = ['product_name', 'aisle', 'department']
existing_drop_cols = [col for col in drop_cols if col in df.columns]
df = df.drop(columns=existing_drop_cols)

# 정수형 컬럼 다운캐스팅
int_cols = [
    'user_id', 'product_id', 'label',
    'up_order_count', 'up_first_order_num', 'up_last_order_num',
    'up_reordered_sum', 'user_total_orders', 'user_total_items',
    'user_total_distinct_items', 'product_total_purchases',
    'product_total_reorders', 'product_unique_users',
    'aisle_id', 'department_id', 'up_recency', 'up_first_last_gap'
]

for col in int_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], downcast='integer')

# 실수형 컬럼 다운캐스팅
float_cols = [
    'up_avg_add_to_cart',
    'user_avg_days_since_prior',
    'product_reorder_rate',
    'up_order_rate'
]

for col in float_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], downcast='float')

# -----------------------------
# 15. CSV 저장
# -----------------------------
df.to_csv(
    '../data/reorder_model_dataset_small.csv',
    index=False,
    encoding='utf-8-sig',
    float_format='%.4f'
)

print("전처리 및 최적화 완료")
print("최종 데이터 크기:", df.shape)
print(df.dtypes)
print(df.head())